# FraudLens: hyperparameter tuning on Kaggle

Tunes **XGBoost** and **LightGBM** with Optuna (about 50 trials each, with pruning), maximising
**validation PR-AUC**, and writes `best_params.json`. Runs on a free Kaggle notebook because the
laptop has 8 GB of RAM.

**Reproducibility:**
- The input is the private dataset built by `fraudlens kaggle-bundle`: train and validation rows
  only (the test set is never uploaded), plus the `fraudlens` wheel, so every trial uses exactly
  the same pipelines and metric code as local training.
- The split dates below are copied from `configs/config.yaml` and checked against the bundle's
  manifest; the data file's SHA-256 is checked too.
- Model libraries are pinned to the laptop's versions; TPE sampling is seeded.

**Setup on Kaggle:** add the dataset (*Add Input*), turn **Internet on** (Settings, needed for
`pip install`), choose the **CPU** accelerator, then *Run All*. See `docs/kaggle_tuning.md`.

In [ ]:
# ---- Settings (split dates copied from configs/config.yaml) ----------------------------
TRAIN_END = "2020-04-21"
VALIDATION_END = "2020-06-21"
SEED = 42
N_TRIALS = 50  # per model
TIMEOUT_PER_MODEL_MIN = 150  # safety cap per model; Kaggle sessions last 12 hours
PINNED = {  # must match the laptop (see manifest.json local_versions)
    "scikit-learn": "1.9.1",
    "xgboost": "3.2.0",
    "lightgbm": "4.7.0",
    "optuna": "5.0.0",
}

import os
from pathlib import Path

SMOKE_TEST = os.environ.get("FRAUDLENS_KAGGLE_SMOKE") == "1"  # local check: 2 tiny trials
if SMOKE_TEST:
    N_TRIALS, TIMEOUT_PER_MODEL_MIN = 2, 5
INPUT_ROOT = Path(os.environ.get("FRAUDLENS_KAGGLE_INPUT", "/kaggle/input"))
OUTPUT_DIR = Path(os.environ.get("FRAUDLENS_KAGGLE_OUTPUT", "/kaggle/working"))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Kaggle mounts datasets under /kaggle/input/<...>/; find the bundle by its manifest.
manifests = sorted(INPUT_ROOT.rglob("manifest.json"))
assert manifests, (
    f"No manifest.json under {INPUT_ROOT}: add the fraudlens-features dataset as input"
)
BUNDLE = manifests[0].parent
print("bundle:", BUNDLE, "| smoke test:", SMOKE_TEST)

In [ ]:
# ---- Install pinned model libraries and the fraudlens wheel -----------------------------
import importlib.metadata as md
import subprocess
import sys

wheel = next(BUNDLE.glob("fraudlens-*.whl"))
need = [f"{p}=={v}" for p, v in PINNED.items() if md.version(p) != v] if not SMOKE_TEST else []
if need:
    print("installing:", need)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *need])
if not SMOKE_TEST:  # locally, fraudlens is already installed (editable) in the project venv
    # --no-deps: the wheel's CLI dependencies are not needed for tuning. The package is pure
    # Python, so it also runs if Kaggle's image is newer than the project's Python 3.11.
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "--no-deps",
            "--force-reinstall",
            "--ignore-requires-python",
            str(wheel),
        ]
    )
versions = {
    "python": sys.version.split()[0],
    **{p: md.version(p) for p in [*PINNED, "pandas", "numpy", "fraudlens"]},
}
print(versions)
mismatch = {p: versions[p] for p in PINNED if versions[p] != PINNED[p]}
assert not mismatch, f"Versions differ from the laptop: {mismatch}. Is Internet on?"

In [ ]:
# ---- Verify the data is exactly what the laptop exported ---------------------------------
import hashlib
import json

import pandas as pd

manifest = json.loads((BUNDLE / "manifest.json").read_text())
assert manifest["split"] == {"train_end": TRAIN_END, "validation_end": VALIDATION_END}, manifest[
    "split"
]
assert manifest["seed"] == SEED

data_path = BUNDLE / manifest["data_file"]
digest = hashlib.sha256(data_path.read_bytes()).hexdigest()
assert digest == manifest["data_sha256"], "data file differs from the manifest"

df = pd.read_parquet(data_path)
assert set(df["split"].unique()) == {"train", "validation"}, "only train/validation may be uploaded"
counts = df.groupby("split")["is_fraud"].agg(["size", "sum"])
assert counts["size"].to_dict() == manifest["rows"] and counts["sum"].to_dict() == manifest["fraud"]
print(counts)

In [ ]:
# ---- Model inputs, using the same function as local training -----------------------------
import logging

from fraudlens.models.estimators import prepare_features
from fraudlens.models.tuning import best_params_payload, tune

logging.basicConfig(
    level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s", force=True
)
import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)  # one summary line per study, not per trial

train, val = df[df["split"] == "train"], df[df["split"] == "validation"]
if SMOKE_TEST:  # tiny, time-ordered subsets for a quick local check of the notebook
    train, val = train.tail(60_000), val.head(30_000)
X_train, y_train = prepare_features(train), train["is_fraud"].to_numpy()
X_val, y_val = prepare_features(val), val["is_fraud"].to_numpy()
print(
    f"train {len(X_train):,} rows ({y_train.sum():,} fraud) | validation {len(X_val):,} rows ({y_val.sum():,} fraud)"
)

In [ ]:
# ---- Tune XGBoost -----------------------------------------------------------------------
studies = {}
studies["xgboost"] = tune(
    "xgboost",
    X_train,
    y_train,
    X_val,
    y_val,
    n_trials=N_TRIALS,
    timeout_seconds=TIMEOUT_PER_MODEL_MIN * 60,
    seed=SEED,
)
studies["xgboost"].trials_dataframe().sort_values("value", ascending=False).head(10)

In [ ]:
# ---- Tune LightGBM ----------------------------------------------------------------------
studies["lightgbm"] = tune(
    "lightgbm",
    X_train,
    y_train,
    X_val,
    y_val,
    n_trials=N_TRIALS,
    timeout_seconds=TIMEOUT_PER_MODEL_MIN * 60,
    seed=SEED,
)
studies["lightgbm"].trials_dataframe().sort_values("value", ascending=False).head(10)

In [ ]:
# ---- Save best_params.json (download it into configs/) and the trial logs ----------------
payload = best_params_payload(
    studies,
    meta={
        "source": "kaggle" if not SMOKE_TEST else "local smoke test",
        "n_trials_requested": N_TRIALS,
        "seed": SEED,
        "split": manifest["split"],
        "data_sha256": manifest["data_sha256"],
        "versions": versions,
    },
)
(OUTPUT_DIR / "best_params.json").write_text(json.dumps(payload, indent=2))
for name, study in studies.items():
    study.trials_dataframe().to_csv(OUTPUT_DIR / f"trials_{name}.csv", index=False)
print(json.dumps({k: v for k, v in payload.items() if k != "_meta"}, indent=2))
print(json.dumps(payload["_meta"]["studies"], indent=2))

## Next steps (on the laptop)

1. In the notebook's **Output** panel, download `best_params.json`.
2. Save it as `configs/best_params.json` in the project (it is committed to Git).
3. Phase 6 retrains with these parameters: `uv run fraudlens train --use-best-params`.

The validation PR-AUC printed here was used to *choose* the parameters, so it is slightly
optimistic; the untouched test set gives the honest final number in Phase 6.